# MS-PlateNet OCR Notebook

Cell-by-cell workflow: Fast Plate OCR, EasyOCR, RapidOCR, PaddleOCR, then combined comparison table.

GPU is used automatically where available (EasyOCR, RapidOCR, PaddleOCR, and the YOLO detector all auto-detect CUDA and fall back to CPU if no GPU is found).

In [1]:
# RapidOCR (ONNXRuntime-based -- works on CPU out of the box; for GPU it
# uses onnxruntime-gpu if installed, otherwise silently uses CPU).
!pip install rapidocr onnxruntime --break-system-packages

# Uncomment the line below INSTEAD of onnxruntime above if you have a
# CUDA-capable GPU + matching CUDA/cuDNN installed, to let RapidOCR run on GPU:
!pip install rapidocr onnxruntime-gpu --break-system-packages


In [3]:
# PaddleOCR + PaddlePaddle.
#
# IMPORTANT: paddlepaddle==3.3.0 has a confirmed bug that breaks CPU
# inference for every PaddleOCR pipeline model with:
#   NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute ...
# (see https://github.com/PaddlePaddle/Paddle/issues/77340). We pin 3.2.2
# below, which is the version before the regression and is confirmed working.
# If a newer 3.3.x patch release fixes this in the future, you can drop the
# version pin.
!pip install paddleocr --break-system-packages
!pip install "paddlepaddle==3.2.2" --break-system-packages

# GPU build alternative (pick the command matching your installed CUDA version,
# see https://www.paddlepaddle.org.cn/en/install/quick for the right index URL):
# !pip install "paddlepaddle-gpu==3.2.2" --break-system-packages


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.2/101.7 MB 4.6 MB/s eta 0:00:23
    --------------------------------------- 2.0/101.7 MB 21.2 MB/s eta 0:00:05
   -- ------------------------------------- 7.1/101.7 MB 50.6 MB/s eta 0:00:02
   --- ------------------------------------ 8.6/101.7 MB 45.5 MB/s eta 0:00:03
   --- ------------------------------------ 9.2/101.7 MB 39.5 MB/s eta 0:00:03
   ---- ----------------------------------- 10.6/101.7 MB 46.9 MB/s eta 0:00:02
   ---- ----------------------------------- 12.2/101.7 MB 43.7 MB/s eta 0:00:03
   ----- ---------------------------------- 15.2/101.7 MB 40.9 MB/s eta 0:00:03
   -------- ------------------------------- 20.7/101.7 MB 72.6 MB/s eta 0:00:02
   --------- ----------------------------- 26.0/101.7 MB 108.8 MB/s eta 0:00:01
   ------------ -------------------------- 31.4/101.7 MB 108.8 MB/s eta 0:00:01
   -------------- ------------------------ 36.6/101.7 M

  You can safely remove it manually.
  You can safely remove it manually.


In [5]:
import cv2, re, time
import numpy as np
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher

In [7]:
# --- GPU detection (shared by every OCR engine below) ---------------------
# We try to ask torch (if installed) whether CUDA is available. If torch
# isn't installed, or it says no, we treat the machine as CPU-only. Each
# engine below still wraps its own GPU init in a try/except and falls back
# to CPU on its own if GPU init fails for an engine-specific reason (e.g.
# wrong onnxruntime-gpu/CUDA version, paddle CPU-only build, etc.) -- this
# flag is just an optimistic first guess, not a hard guarantee.
GPU_AVAILABLE = False
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
except Exception:
    GPU_AVAILABLE = False

print(f'[gpu] GPU_AVAILABLE = {GPU_AVAILABLE}')

try:
    from ultralytics import YOLO
except Exception as e:
    YOLO = None
    print(f'[import] ultralytics unavailable: {type(e).__name__}: {e}')

# FIX: class is named LicensePlateRecognizer in current fast-plate-ocr releases
# (ONNXPlateRecognizer was the old name and no longer exists).
try:
    from fast_plate_ocr import LicensePlateRecognizer
except Exception as e:
    LicensePlateRecognizer = None
    print(f'[import] fast_plate_ocr unavailable: {type(e).__name__}: {e}')

try:
    import easyocr
except Exception as e:
    easyocr = None
    print(f'[import] easyocr unavailable: {type(e).__name__}: {e}')

try:
    from rapidocr import RapidOCR
except Exception as e:
    try:
        # older package name: rapidocr_onnxruntime
        from rapidocr_onnxruntime import RapidOCR
    except Exception as e2:
        RapidOCR = None
        print(f'[import] rapidocr unavailable: {type(e).__name__}: {e}')

try:
    from paddleocr import PaddleOCR
except Exception as e:
    PaddleOCR = None
    print(f'[import] paddleocr unavailable: {type(e).__name__}: {e}')


[gpu] GPU_AVAILABLE = True


Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [35]:
BASE = Path.cwd()
IMG_DIR = BASE / 'ocr_test_images'
GT_CSV = BASE / 'ground_truth.csv'
OUT_DIR = BASE / 'output'
OUT_DIR.mkdir(exist_ok=True)
MODEL_PATH = Path("Version-5-runs/plateNetV5medium/weights/bestMediumV5.pt")

gt = pd.read_csv(GT_CSV)
img_paths = sorted([p for p in IMG_DIR.glob('*') if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']])[:20]

def clean(s):
    return re.sub(r'[^A-Z0-9]', '', str(s).upper()) if s is not None else ''

def sim(a, b):
    return SequenceMatcher(None, clean(a), clean(b)).ratio()

In [37]:
if YOLO is None:
    raise ImportError('ultralytics is not installed')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

model = YOLO(str(MODEL_PATH))

# FIX: device=0 hardcoded GPU index 0 and would crash with no CUDA device
# present. Use the shared GPU_AVAILABLE flag (computed in the imports cell)
# to run on GPU when one is available and fall back to CPU otherwise.
YOLO_DEVICE = 0 if GPU_AVAILABLE else 'cpu'
print(f'[yolo] running detector on device={YOLO_DEVICE!r}')

def detect_plate_crop(img_bgr):
    results = model.predict(img_bgr, conf=0.25, verbose=False, device=YOLO_DEVICE)
    if not results or len(results) == 0:
        return None, None
    r = results[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None, None
    boxes = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()
    i = int(np.argmax(confs))
    x1, y1, x2, y2 = boxes[i].astype(int)
    h, w = img_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return None, None
    return img_bgr[y1:y2, x1:x2].copy(), float(confs[i])


[yolo] running detector on device=0


## First cell: Fast Plate OCR

In [40]:
# FIX: no .predict() method exists on LicensePlateRecognizer -- use .run_one()
# for a single image. It returns a PlatePrediction object/dict with a `.plate`
# field, not the text/result/prediction keys the old code was guessing at.
# FIX: must pass a hub model name (or onnx_model_path); calling with no args
# does not just "use a default" the way the old code assumed.

fast_model = None
if LicensePlateRecognizer is not None:
    try:
        # Pick the hub model matching your plates. Options include:
        # 'cct-xs-v1-global-model', 'cct-s-v2-global-model',
        # 'european-plates-mobile-vit-v2-model', 'global-plates-mobile-vit-v2-model', etc.
        fast_model = LicensePlateRecognizer('cct-xs-v1-global-model')
    except Exception as e:
        print(f'[fast_plate_ocr init failed] {type(e).__name__}: {e}')
        fast_model = None
else:
    print('[fast_plate_ocr] not available -- see import error above')

fast_rows = []
for p in img_paths:
    truth_row = gt.loc[gt['image'] == p.name, 'plate']
    truth = clean(truth_row.iloc[0]) if len(truth_row) else ''
    img_bgr = cv2.imread(str(p))
    crop, det_conf = detect_plate_crop(img_bgr)

    pred = ''
    fast_raw = ''
    if fast_model is not None and crop is not None:
        try:
            out = fast_model.run_one(crop)
            fast_raw = str(out)
            plate_text = getattr(out, 'plate', None)
            if plate_text is None and isinstance(out, dict):
                plate_text = out.get('plate')
            pred = clean(plate_text) if plate_text is not None else ''
        except Exception as e:
            fast_raw = f'ERR: {type(e).__name__}: {e}'
            pred = ''
    elif fast_model is None:
        fast_raw = 'ERR: fast_model not initialized'
    elif crop is None:
        fast_raw = 'ERR: no plate detected by YOLO'

    fast_rows.append({
        'image': p.name,
        'truth': truth,
        'fast_raw': fast_raw,
        'fast_plate_ocr': pred,
        'fast_ok': int(pred == truth),
        'fast_sim': round(sim(pred, truth), 4),
        'det_conf': det_conf
    })

fast_df = pd.DataFrame(fast_rows)
fast_df.to_csv(OUT_DIR / 'fast_plate_ocr_results.csv', index=False)
fast_df


,image,truth,fast_raw,fast_plate_ocr,fast_ok,fast_sim,det_conf
0,img1.jpg,MCY6055,"PlatePrediction(plate='MCY6055', char_probs=No...",MCY6055,1,1.0000,0.819638
1,img10.jpg,SJS4878P,"PlatePrediction(plate='SJS4878P', char_probs=N...",SJS4878P,1,1.0000,0.731283
2,img11.jpg,SMR54X,"PlatePrediction(plate='SMR54X', char_probs=Non...",SMR54X,1,1.0000,0.783089
3,img12.jpg,WJJ2495,"PlatePrediction(plate='WJJ2495', char_probs=No...",WJJ2495,1,1.0000,0.861307
4,img13.jpg,PCM5078,"PlatePrediction(plate='PCM5078', char_probs=No...",PCM5078,1,1.0000,0.866372
5,img14.jpg,BRA8686,"PlatePrediction(plate='BRA8686', char_probs=No...",BRA8686,1,1.0000,0.823842
6,img15.jpg,WB3233T,"PlatePrediction(plate='NB22333', char_probs=No...",NB22333,0,0.5714,0.755026
7,img16.jpg,ACD128,"PlatePrediction(plate='AC0128', char_probs=Non...",AC0128,0,0.8333,0.576379
8,img17.jpg,VKK4039,"PlatePrediction(plate='VKK4039', char_probs=No...",VKK4039,1,1.0000,0.885560
9,img18.jpg,NBC9482,"PlatePrediction(plate='NBC9482', char_probs=No...",NBC9482,1,1.0000,0.861166


## Second cell: EasyOCR

In [43]:
easy = None
if easyocr is not None:
    try:
        easy = easyocr.Reader(['en'], gpu=GPU_AVAILABLE)
    except Exception as e:
        print(f'[easyocr GPU init failed, retrying on CPU] {type(e).__name__}: {e}')
        try:
            easy = easyocr.Reader(['en'], gpu=False)
        except Exception as e2:
            print(f'[easyocr CPU init also failed] {type(e2).__name__}: {e2}')
            easy = None
else:
    print('[easyocr] not available -- see import error above')

easy_rows = []
for p in img_paths:
    truth_row = gt.loc[gt['image'] == p.name, 'plate']
    truth = clean(truth_row.iloc[0]) if len(truth_row) else ''
    img_bgr = cv2.imread(str(p))
    crop, det_conf = detect_plate_crop(img_bgr)

    pred = ''
    easy_raw = ''
    if easy is not None and crop is not None:
        try:
            texts = easy.readtext(crop, detail=0, paragraph=False)
            easy_raw = str(texts)
            # NOTE: readtext() returning [] (no text detected) is a legitimate
            # "no result" -- ''.join([]) silently becomes '', which previously
            # looked identical to a crash. easy_raw above lets you tell the
            # difference: '[]' means EasyOCR ran but found nothing.
            pred = clean(''.join(texts))
        except Exception as e:
            easy_raw = f'ERR: {type(e).__name__}: {e}'
            pred = ''
    elif easy is None:
        easy_raw = 'ERR: easyocr reader not initialized'
    elif crop is None:
        easy_raw = 'ERR: no plate detected by YOLO'

    easy_rows.append({
        'image': p.name,
        'truth': truth,
        'easy_raw': easy_raw,
        'easyocr': pred,
        'easy_ok': int(pred == truth),
        'easy_sim': round(sim(pred, truth), 4),
        'det_conf': det_conf
    })

easy_df = pd.DataFrame(easy_rows)
easy_df.to_csv(OUT_DIR / 'easyocr_results.csv', index=False)
easy_df


,image,truth,easy_raw,easyocr,easy_ok,easy_sim,det_conf
0,img1.jpg,MCY6055,['McY 6055'],MCY6055,1,1.0000,0.819638
1,img10.jpg,SJS4878P,['SJS4878P'],SJS4878P,1,1.0000,0.731283
2,img11.jpg,SMR54X,['SMREAX'],SMREAX,0,0.6667,0.783089
3,img12.jpg,WJJ2495,"['WJJ', '2495']",WJJ2495,1,1.0000,0.861307
4,img13.jpg,PCM5078,"['75078', 'PCM']",75078PCM,0,0.5333,0.866372
5,img14.jpg,BRA8686,['BRA86O6'],BRA86O6,0,0.8571,0.823842
6,img15.jpg,WB3233T,['WB32331'],WB32331,0,0.8571,0.755026
7,img16.jpg,ACD128,['AcDPB'],ACDPB,0,0.5455,0.576379
8,img17.jpg,VKK4039,"['VKK', '4039', 'Totota""', 'Troton""']",VKK4039TOTOTATROTON,0,0.5385,0.885560
9,img18.jpg,NBC9482,['NBC 9482'],NBC9482,1,1.0000,0.861166


## Third cell: RapidOCR

In [46]:
# RapidOCR auto-detects GPU through onnxruntime: if onnxruntime-gpu is
# installed (and a working CUDA/cuDNN setup is found), it uses GPU; otherwise
# it transparently falls back to the CPU onnxruntime build. We just try GPU
# params first and fall back to default (CPU) params if that fails.

rapid_engine = None
if RapidOCR is not None:
    try:
        if GPU_AVAILABLE:
            rapid_engine = RapidOCR(params={"EngineConfig.onnxruntime.use_cuda": True})
        else:
            rapid_engine = RapidOCR()
    except Exception as e:
        print(f'[rapidocr GPU init failed, retrying on CPU] {type(e).__name__}: {e}')
        try:
            rapid_engine = RapidOCR()
        except Exception as e2:
            print(f'[rapidocr CPU init also failed] {type(e2).__name__}: {e2}')
            rapid_engine = None
else:
    print('[rapidocr] not available -- see import error above')

rapid_rows = []
for p in img_paths:
    truth_row = gt.loc[gt['image'] == p.name, 'plate']
    truth = clean(truth_row.iloc[0]) if len(truth_row) else ''
    img_bgr = cv2.imread(str(p))
    crop, det_conf = detect_plate_crop(img_bgr)

    pred = ''
    rapid_raw = ''
    if rapid_engine is not None and crop is not None:
        try:
            result = rapid_engine(crop)
            # RapidOCR returns a result object; .txts holds the recognized
            # strings (newer API) -- fall back to indexing [0] (older API:
            # list of [box, text, score]) if .txts isn't present.
            texts = getattr(result, 'txts', None)
            if texts is None and result:
                texts = [line[1] for line in result[0]] if result[0] else []
            texts = texts or []
            rapid_raw = str(texts)
            pred = clean(''.join(texts))
        except Exception as e:
            rapid_raw = f'ERR: {type(e).__name__}: {e}'
            pred = ''
    elif rapid_engine is None:
        rapid_raw = 'ERR: rapidocr engine not initialized'
    elif crop is None:
        rapid_raw = 'ERR: no plate detected by YOLO'

    rapid_rows.append({
        'image': p.name,
        'truth': truth,
        'rapid_raw': rapid_raw,
        'rapidocr': pred,
        'rapid_ok': int(pred == truth),
        'rapid_sim': round(sim(pred, truth), 4),
        'det_conf': det_conf
    })

rapid_df = pd.DataFrame(rapid_rows)
rapid_df.to_csv(OUT_DIR / 'rapidocr_results.csv', index=False)
rapid_df


[INFO] 2026-06-22 02:26:18,702 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-22 02:26:18,721 [RapidOCR] download_file.py:60: File exists and is valid: C:\Software\anaconda3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-22 02:26:18,723 [RapidOCR] main.py:65: Using C:\Software\anaconda3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[WARNING] 2026-06-22 02:26:18,723 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[INFO] 2026-06-22 02:26:18,724 [RapidOCR] provider_config.py:203: If you want to use CUDAExecutionProvider acceleration, you must do:(For reference only) If you want to use GPU acceleration, you must do:
[INFO] 2026-06-22 02:26:18,724 [RapidOCR] provider_config.py:203: First, uninstall all onnxruntime packages in current environment.
[INFO] 2026-06-22 02:26:18,725 [RapidO

,image,truth,rapid_raw,rapidocr,rapid_ok,rapid_sim,det_conf
0,img1.jpg,MCY6055,"('MCY', '6055')",MCY6055,1,1.0000,0.819638
1,img10.jpg,SJS4878P,"('Sr', '4', '8', '76', '8', 'JL')",SR48768JL,0,0.5882,0.731283
2,img11.jpg,SMR54X,"('SMR5', '54', '4X')",SMR5544X,0,0.8571,0.783089
3,img12.jpg,WJJ2495,"('WJJ', '2495')",WJJ2495,1,1.0000,0.861307
4,img13.jpg,PCM5078,"('PCM5078',)",PCM5078,1,1.0000,0.866372
5,img14.jpg,BRA8686,"('BRA8686',)",BRA8686,1,1.0000,0.823842
6,img15.jpg,WB3233T,[],,0,0.0000,0.755026
7,img16.jpg,ACD128,[],,0,0.0000,0.576379
8,img17.jpg,VKK4039,"('VKK4039', 'UMW TOYOTA NOTOR')",VKK4039UMWTOYOTANOTOR,0,0.5000,0.885560
9,img18.jpg,NBC9482,"('NBC', 'C9482')",NBCC9482,0,0.9333,0.861166


## Fourth cell: PaddleOCR

In [49]:
# PaddleOCR 3.x removed use_angle_cls / show_log / use_gpu from the
# constructor (show_log now hard-errors instead of warning, which is why
# init was failing). GPU/CPU is now picked with device='gpu'/'cpu', and the
# angle-classifier flag is now use_textline_orientation. We try the 3.x
# signature first and fall back to the legacy 2.x signature if that's what's
# actually installed.

paddle_engine = None
paddle_api = None  # 'v3' or 'v2', so the inference loop knows how to call it
if PaddleOCR is not None:
    device = 'gpu' if GPU_AVAILABLE else 'cpu'
    try:
        # PaddleOCR 3.x signature
        paddle_engine = PaddleOCR(use_textline_orientation=True, lang='en', device=device)
        paddle_api = 'v3'
    except TypeError as e:
        # TypeError here means the installed version doesn't recognize these
        # kwargs at all -- i.e. it's the older 2.x API instead.
        print(f'[paddleocr] v3-style init failed ({type(e).__name__}: {e}), trying legacy v2 API')
        try:
            paddle_engine = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=GPU_AVAILABLE)
            paddle_api = 'v2'
        except Exception as e2:
            print(f'[paddleocr legacy init also failed] {type(e2).__name__}: {e2}')
            paddle_engine = None
    except Exception as e:
        # GPU device requested but not actually usable -> retry on CPU.
        print(f'[paddleocr GPU init failed, retrying on CPU] {type(e).__name__}: {e}')
        try:
            paddle_engine = PaddleOCR(use_textline_orientation=True, lang='en', device='cpu')
            paddle_api = 'v3'
        except Exception as e2:
            print(f'[paddleocr CPU init also failed] {type(e2).__name__}: {e2}')
            paddle_engine = None
else:
    print('[paddleocr] not available -- see import error above')

paddle_rows = []
for p in img_paths:
    truth_row = gt.loc[gt['image'] == p.name, 'plate']
    truth = clean(truth_row.iloc[0]) if len(truth_row) else ''
    img_bgr = cv2.imread(str(p))
    crop, det_conf = detect_plate_crop(img_bgr)

    pred = ''
    paddle_raw = ''
    if paddle_engine is not None and crop is not None:
        try:
            texts = []
            if paddle_api == 'v3':
                # .predict() returns a list of result objects (dict-like),
                # one per input image; each has a 'rec_texts' list.
                results = paddle_engine.predict(crop)
                paddle_raw = str(results)
                if results:
                    res0 = results[0]
                    texts = res0.get('rec_texts', []) if hasattr(res0, 'get') else getattr(res0, 'rec_texts', [])
            else:
                # legacy .ocr() -> list of [box, (text, score)] per image
                result = paddle_engine.ocr(crop, cls=True)
                paddle_raw = str(result)
                if result and result[0]:
                    texts = [line[1][0] for line in result[0]]
            texts = texts or []
            pred = clean(''.join(texts))
        except Exception as e:
            paddle_raw = f'ERR: {type(e).__name__}: {e}'
            pred = ''
    elif paddle_engine is None:
        paddle_raw = 'ERR: paddleocr engine not initialized'
    elif crop is None:
        paddle_raw = 'ERR: no plate detected by YOLO'

    paddle_rows.append({
        'image': p.name,
        'truth': truth,
        'paddle_raw': paddle_raw,
        'paddleocr': pred,
        'paddle_ok': int(pred == truth),
        'paddle_sim': round(sim(pred, truth), 4),
        'det_conf': det_conf
    })

paddle_df = pd.DataFrame(paddle_rows)
paddle_df.to_csv(OUT_DIR / 'paddleocr_results.csv', index=False)
paddle_df


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\USER 1\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\USER 1\.paddlex\official_models\UVDoc`.
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\USER 1\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\USER 1\.pad

,image,truth,paddle_raw,paddleocr,paddle_ok,paddle_sim,det_conf
0,img1.jpg,MCY6055,"[{'input_path': None, 'page_index': None, 'doc...",MCY6055,1,1.0000,0.819638
1,img10.jpg,SJS4878P,"[{'input_path': None, 'page_index': None, 'doc...",SJS4878P,1,1.0000,0.731283
2,img11.jpg,SMR54X,"[{'input_path': None, 'page_index': None, 'doc...",SMR54X,1,1.0000,0.783089
3,img12.jpg,WJJ2495,"[{'input_path': None, 'page_index': None, 'doc...",2495TCM,0,0.5714,0.861307
4,img13.jpg,PCM5078,"[{'input_path': None, 'page_index': None, 'doc...",PCM5078,1,1.0000,0.866372
5,img14.jpg,BRA8686,"[{'input_path': None, 'page_index': None, 'doc...",8RA8686,0,0.8571,0.823842
6,img15.jpg,WB3233T,"[{'input_path': None, 'page_index': None, 'doc...",VB3233,0,0.7692,0.755026
7,img16.jpg,ACD128,"[{'input_path': None, 'page_index': None, 'doc...",ACD128,1,1.0000,0.576379
8,img17.jpg,VKK4039,"[{'input_path': None, 'page_index': None, 'doc...",VKK4039UMWTOYOTAMOTOR,0,0.5000,0.885560
9,img18.jpg,NBC9482,"[{'input_path': None, 'page_index': None, 'doc...",NBC9482,1,1.0000,0.861166


## Combined comparison table

In [51]:
combined = pd.DataFrame({'image': gt['image'].astype(str), 'truth': gt['plate'].astype(str)})
combined = combined.merge(fast_df[['image','fast_plate_ocr','fast_ok','fast_sim']], on='image', how='left')
combined = combined.merge(easy_df[['image','easyocr','easy_ok','easy_sim']], on='image', how='left')
combined = combined.merge(rapid_df[['image','rapidocr','rapid_ok','rapid_sim']], on='image', how='left')
combined = combined.merge(paddle_df[['image','paddleocr','paddle_ok','paddle_sim']], on='image', how='left')
combined.to_csv(OUT_DIR / 'ocr_combined_comparison.csv', index=False)

summary = pd.DataFrame({
    'OCR': ['fast_plate_ocr', 'easyocr', 'rapidocr', 'paddleocr'],
    'ExactMatchAcc': [combined['fast_ok'].mean(), combined['easy_ok'].mean(), combined['rapid_ok'].mean(), combined['paddle_ok'].mean()],
    'AvgSim': [combined['fast_sim'].mean(), combined['easy_sim'].mean(), combined['rapid_sim'].mean(), combined['paddle_sim'].mean()]
})
summary.to_csv(OUT_DIR / 'ocr_summary.csv', index=False)
combined


,image,truth,fast_plate_ocr,fast_ok,fast_sim,easyocr,easy_ok,easy_sim,rapidocr,rapid_ok,rapid_sim,paddleocr,paddle_ok,paddle_sim
0,img1.jpg,MCY6055,MCY6055,1,1.0000,MCY6055,1,1.0000,MCY6055,1,1.0000,MCY6055,1,1.0000
1,img2.jpg,WHN9050,WHN9050,1,1.0000,WHN9050,1,1.0000,WHN9050,1,1.0000,9050NHM,0,0.5714
2,img3.jpg,WRA73,WRA73,1,1.0000,MRA73,0,0.8000,WRA473,0,0.9091,WRA73,1,1.0000
3,img4.jpg,NEG8531,NEG8531,1,1.0000,NEG8531,1,1.0000,NEG8531,1,1.0000,G8531NEG,0,0.6667
4,img5.jpg,B87777NRS,B87777NRS,1,1.0000,B87777NRS,1,1.0000,B887777NRS,0,0.9474,B87777NRS,1,1.0000
5,img6.jpg,FA555,FA555,1,1.0000,FA535,0,0.8000,FA555,1,1.0000,MASS,0,0.2222
6,img7.jpg,VJF458,VJF458,1,1.0000,WUF458,0,0.6667,WJF458,0,0.8333,RA7,0,0.0000
7,img8.jpg,SJA2099D,SJA2099D,1,1.0000,SJA20990,0,0.8750,SJJA2099D,0,0.9412,20990SJA,0,0.5000
8,img9.jpg,SNW9359U,AMM9358U,0,0.5000,EN93590,0,0.6667,,0,0.0000,,0,0.0000
9,img10.jpg,SJS4878P,SJS4878P,1,1.0000,SJS4878P,1,1.0000,SR48768JL,0,0.5882,SJS4878P,1,1.0000


## Summary table

In [ ]:
summary